In [1]:
from lib import *

ImportError: Unable to import required dependencies:
numpy: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

In [3]:
import os
print(os.getcwd())

d:\galaxycine_data\glx-data\glx_data\payments


Create function allocated for payment

In [2]:
def allocate_payments(transactions, payment_methods):
    # Initialize balances for payment methods
    balances = payment_methods.copy()
    allocation = []

    for transaction in transactions:
        transaction_allocation = [0] * len(balances)  # Initialize allocation for this transaction
        remaining_amount = transaction  # Remaining amount to be paid for this transaction

        for i, balance in enumerate(balances):
            if abs(remaining_amount) <= balance:  # Can pay fully from this method
                transaction_allocation[i] = remaining_amount
                balances[i] -= remaining_amount
                remaining_amount = 0
                break
            else:  # Use up all balance in this method
                transaction_allocation[i] = balance
                remaining_amount -= balance
                balances[i] = 0

        if remaining_amount > 0:
            print("Warning: Not enough funds to cover the transaction!")
        
        allocation.append(transaction_allocation)

    return allocation


**Receipt BOCO Detail**

In [3]:
with engine_site_any.connect() as conn:
    sql_query = pd.read_sql_query('''
                                DECLARE @fromdate datetime,
                                    @todate datetime,
                                    @closedbusinesstime datetime,
                                    @closedtime datetime
                                    SET
                                    @closedbusinesstime = '2025-04-28 00:00:00.000'
                                    --DATEADD(HOUR, 0, CONVERT(VARCHAR(10), GETDATE()-15, 110))
                                    SET
                                    @fromdate = '2025-04-28 00:00:00.000'
                                    --DATEADD(HOUR, 6, CONVERT(VARCHAR(10), GETDATE()-15, 110))
                                    SET
                                    @todate = '2025-05-05 00:00:00.000'
                                    --DATEADD(HOUR, 6, CONVERT(VARCHAR(10), GETDATE()-14, 110))
                                    -- Get Closing Date
                                    SELECT
                                                        @closedtime = DCJ_dtmCreated
                                                    FROM
                                                        tblDailyCashJournal
                                                    WHERE
                                                        DCJ_dtmBusinessDate = @closedbusinesstime
                                                        AND DCJ_strStatus = 'C'
                                                        -- Main query
                                                                                                SELECT
                                                        right(Cin.CinOperator_strCode,
                                                                                                    4) as cinemaCode,
                                                        T.TransT_lgnNumber as transactionNumber,
                                                        T.TransT_intSequence as transactionSeq_Detail,
                                                        T.TransT_intSeqRefunded as transactionRefundSeq,
                                                        @closedtime as closingDate,
                                                        Session_dtmShowing as redeemDate,
                                                        T.TransT_dtmDateTime as transactionDate,
                                                        T.TransT_dtmRealTransTime as realTransactionDate,
                                                        F.Film_strHOFilmCode as productHOCode,
                                                        F.Film_strTitle as productName,
                                                        'BO' as typeProduct,
                                                        TT.TType_strHOCode itemHOCode,
                                                        TT.TType_strDescription as itemName,
                                                        T.TransT_intNoOfSeats as quantity,
                                                        T.TransT_curValueEach as unitPrice,
                                                        'Ticket' as donViTinh,
                                                        (ISNULL( ( CASE
                                                        WHEN T.TransT_curRedempValueEach = 0 THEN TransT_intNoOfSeats * TransT_curValueEach
                                                        ELSE 0
                                                            END ),
                                                            0 )) as gross_detail,
                                                        (TransT_intNoOfSeats * (TransT_curTaxAmount + ISNULL(TransT_curTaxAmount2,
                                                        0) + ISNULL(TransT_curTaxAmount3,
                                                        0) + ISNULL(TransT_curTaxAmount4,
                                                        0))) as tax,
                                                        ISNULL( ( CASE
                                                        WHEN T.TransT_curRedempValueEach = 0 THEN TransT_intNoOfSeats * ( TransT_curValueEach - TransT_curTaxAmount -
                                                        ISNULL(TransT_curTaxAmount2,
                                                                0) - ISNULL(TransT_curTaxAmount3,
                                                                0) - ISNULL(TransT_curTaxAmount4,
                                                                0) )
                                                                ELSE 0
                                                            END ),
                                                            0) as net,
                                                        case
                                                            when TransT_strType = 'A'
                                                            and TransT_strStatus = 'C' then 'RefundPortion'
                                                            when TransT_strType = 'P'
                                                            and TransT_strStatus = 'V' then 'Sold'
                                                            when TransT_strType = 'P'
                                                            and TransT_strStatus = 'R' then 'TicketRefunded'
                                                        end as status
                                                    FROM
                                                        tblTrans_Ticket T
                                                    INNER JOIN tblTicketType TT ON
                                                        TT.TType_strCode = T.Price_strCode
                                                    INNER JOIN tblCinema_Operator Cin ON
                                                        Cin.CinOperator_strCode = T.CinOperator_strCode
                                                    LEFT JOIN tblPrice P ON
                                                        T.PGroup_strCode = P.PGroup_strCode
                                                        AND T.Price_strCode = P.Price_strCode
                                                    LEFT JOIN tblSession S ON
                                                        T.Session_lngSessionId = S.Session_lngSessionId
                                                    LEFT JOIN tblFilm F ON
                                                        S.Film_strCode = F.Film_strCode
                                                    WHERE
                                                        S.Session_dtmRealShow >= @fromdate
                                                        -- input fromdate
                                                        AND S.Session_dtmRealShow < @todate
                                                    UNION ALL
                                                    -- CO Counter
                                                                                                SELECT
                                                        right(Cin.CinOperator_strCode,
                                                                                                    4) as cinemaCode,
                                                        I.TransI_lgnNumber transactionNumber,
                                                        I.TransI_intSequence transactionSeq_Detail,
                                                        I.TransI_intSeqRefunded transactionRefundSeq,
                                                        @closedtime as closingDate,
                                                        I.TransI_dtmDateTime as redeemDate,
                                                        I.TransI_dtmDateTime as transactionDate,
                                                        I.TransI_dtmRealTransTime as realTransactionDate,
                                                        T.Item_strMasterItemCode as productHOCode,
                                                        Item_strItemDescription as productName,
                                                        IC.Class_strDescription as typeProduct,
                                                        Item_strItemDescription as itemHOCode,
                                                        IC.Class_strDescription as itemName,
                                                        TransI_decNoOfItems as quantity,
                                                        TransI_curValueEach unitPrice,
                                                        UOM.Unit_strDescription as donViTinh,
                                                        (Case
                                                            ISNULL(Item_curSaleUOMConv,
                                                                                                        0) WHEN 0 THEN
                                                            --handle divide by zero
                                                            Round((IsNull(TransI_decNoOfItems, (TransI_decNoOfItems / 1)) * TransI_curValueEach), 6)
                                                            ELSE
                                                            --this is the typical path
                                                            Round((IsNull(TransI_decNoOfItems, (TransI_decNoOfItems / 1)) * TransI_curValueEach), 6)
                                                        END) as gross_detail,
                                                        ROUND(( Round((TransI_decNoOfItems * ISNULL(TransI_curSTaxEach, 0)) , 6) + 
                                                        Round((TransI_decNoOfItems * ISNULL(TransI_curSTaxEach2, 0)) , 6) +
                                                        Round((TransI_decNoOfItems * ISNULL(TransI_curSTaxEach3, 0)) , 6) + 
                                                        Round((TransI_decNoOfItems * ISNULL(TransI_curSTaxEach4, 0)) , 6)    
                                                             ), 2) as tax,
                                                        ROUND((Case Item_curSaleUOMConv WHEN 0 THEN
                                                    (Round((TransI_decNoOfItems * TransI_curValueEach) , 6) - 
                                                    ( Round((TransI_decNoOfItems * ISNULL(TransI_curSTaxEach, 0)) , 6) + 
                                                        Round((TransI_decNoOfItems * ISNULL(TransI_curSTaxEach2, 0)) , 6) +
                                                        Round((TransI_decNoOfItems * ISNULL(TransI_curSTaxEach3, 0)) , 6) + 
                                                        Round((TransI_decNoOfItems * ISNULL(TransI_curSTaxEach4, 0)) , 6)    
                                                    ))
                                                            ELSE
                                                    (Round((TransI_decNoOfItems * TransI_curValueEach) , 6) - 
                                                    ( Round((TransI_decNoOfItems * ISNULL(TransI_curSTaxEach, 0)) , 6) + 
                                                        Round((TransI_decNoOfItems * ISNULL(TransI_curSTaxEach2, 0)) , 6) +
                                                        Round((TransI_decNoOfItems * ISNULL(TransI_curSTaxEach3, 0)) , 6) + 
                                                        Round((TransI_decNoOfItems * ISNULL(TransI_curSTaxEach4, 0)) , 6)    
                                                    ))
                                                END), 2) net,
                                                        case
                                                            when I.TransI_strStatus = 'R' then 'CORefunded'
                                                            else 'Sold'
                                                        end as status
                                                    FROM
                                                        tblTrans_Inventory I
                                                    INNER JOIN tblItem T ON
                                                        I.Item_strItemId = T.Item_strItemId
                                                    INNER JOIN tblItem_Class IC ON
                                                        T.Class_strCode = IC.Class_strCode
                                                    INNER JOIN tblCinema_Operator Cin ON
                                                        Cin.CinOperator_strCode = I.CinOperator_strCode
                                                    INNER JOIN tblSalesTax ST ON
                                                        T.STax_strCode = ST.STax_strCode
                                                    INNER JOIN tblStock_Location SL ON
                                                        I.Location_strCode = SL.Location_strCode
                                                    LEFT JOIN tblUnitOfMeasure UOM ON
                                                        T.Item_strBaseUOMCode = UOM.Unit_strCode
                                                    LEFT JOIN tblSession S ON
                                                        I.TransI_lngSessionId = S.Session_lngSessionId
                                                    WHERE
                                                        1 = 1
                                                        AND I.TransI_strType = 'S'
                                                        AND I.TransI_strStatus <> 'W'
                                                        --AND T.Item_strReport <> 'Y' 
                                                        AND I.TransI_dtmDateTime >= @fromdate
                                                        AND I.TransI_dtmDateTime < @todate
                                                        ''', conn.connection)
df_boco_detail = pd.DataFrame(sql_query)

OperationalError: (pyodbc.OperationalError) ('08001', '[08001] [Microsoft][ODBC Driver 17 for SQL Server]Named Pipes Provider: Could not open a connection to SQL Server [53].  (53) (SQLDriverConnect); [08001] [Microsoft][ODBC Driver 17 for SQL Server]Login timeout expired (0); [08001] [Microsoft][ODBC Driver 17 for SQL Server]A network-related or instance-specific error has occurred while establishing a connection to SQL Server. Server is not found or not accessible. Check if instance name is correct and if SQL Server is configured to allow remote connections. For more information see SQL Server Books Online. (53)')
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [1]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'ODBC Driver 18 for SQL Server', 'ODBC Driver 17 for SQL Server']


In [4]:
# Sorting by 'transactionNumber' in ascending order
sorted_df_boco_detail = df_boco_detail.sort_values(by='transactionNumber')
sorted_df_boco_detail.shape

(17010, 20)

Receipt Cash PaymentType

In [5]:
with engine_site_any.connect() as conn:
    sql_query_cash = pd.read_sql_query(f'''
                                   DECLARE @fromdate datetime,
                                    @todate datetime
                                    SET
                                    @fromdate = '2025-04-28 00:00:00.000'
                                    --DATEADD(HOUR, 6, CONVERT(VARCHAR(10), GETDATE()-15, 110))
                                    SET
                                    @todate = '2025-05-05 00:00:00.000'
                                    --DATEADD(HOUR, 6, CONVERT(VARCHAR(10), GETDATE()-14, 110))
                                    Select
                                        right(cash.CinOperator_strCode,
                                        4) as cinemaCode,
                                        cash.TransC_lgnNumber as transactionNumber,
                                        cash.TransC_intSequence as transactionSeq_Total,
                                        (CASE
                                            WHEN cash.TransC_strBKCardType = ''
                                            OR cash.TransC_strBKCardType IS NULL
                                            OR UPPER(cash.TransC_strBKCardType) = 'OTHER'
                                            OR UPPER(cash.TransC_strBKCardType) = 'UNKNOWN'
                                                            THEN ISNULL((
                                            Select
                                                top 1 UPPER(Card_strCardType)
                                            from
                                                tblCardDefinition
                                            where
                                                Left(cash.TransC_strBKCardNo,
                                                6) >= Card_strStartRange
                                                AND left(cash.TransC_strBKCardNo,
                                                6) < Card_strEndRange
                                                AND Upper(Card_strPaymentCard) = 'Y'
                                                AND (cash.TransC_strBKCardNo <> ''
                                                    OR cash.TransC_strBKCardNo IS NOT NULL)
                                            order by
                                                Card_intRangeSize Asc) ,
                                            'UNKNOWN')
                                            ELSE UPPER(cash.TransC_strBKCardType)
                                        END) as paymentType,
                                        cash.TransC_strPaymentTransRef as paymentTransRef,
                                        cash.TransC_curValue as gross_total
                                    From
                                        tblTrans_Cash cash WITH (INDEX = indDateTime),
                                        tblPaymentType pay
                                    Where
                                        1 = 1
                                        AND (cash.TransC_dtmDateTime >= @fromdate
                                        AND cash.TransC_dtmDateTime < @todate)
                                        AND cash.TransC_strType = pay.PayType_strType
                                        AND 'Y' IN ( pay.PayType_strCreditCard, pay.PayType_strDebitCard )
                                    union all

                                    SELECT
                                        right(cash.CinOperator_strCode,
                                        4) as cinemaCode,
                                        cash.TransC_lgnNumber as transactionNumber,
                                        cash.TransC_intSequence as transactionSeq_Total,
                                        tblPaymentType.PayType_strDescription as paymentType,
                                        cash.TransC_strPaymentTransRef as paymentTransRef,
                                        cash.TransC_curValue as gross_total
                                    FROM
                                        tblTrans_Cash cash WITH (INDEX = indDateTime),
                                        tblPaymentType
                                    WHERE
                                        1 = 1
                                        AND cash.TransC_strType = tblPaymentType.PayType_strType
                                        AND (cash.TransC_dtmDateTime >= @fromdate
                                        AND cash.TransC_dtmDateTime < @todate)
                                        --AND cash.TransC_dtmDateTime BETWEEN @fromdate AND @todate
                                        AND 'Y' NOT IN ( tblPaymentType.PayType_strCreditCard, tblPaymentType.PayType_strDebitCard)
                                    ''', conn.connection)
    
df_receipt_cash_paymentType = pd.DataFrame(sql_query_cash)

C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\482609893.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sql_query_cash = pd.read_sql_query(f'''


In [6]:
sorted_df_paymentType = df_receipt_cash_paymentType.sort_values(by='transactionNumber')
sorted_df_paymentType.shape

(5411, 6)

In [7]:
sorted_df_paymentType = sorted_df_paymentType.rename(columns={'paymentType': ''})
df_receipt_cash_paymentType.columns

Index(['cinemaCode', 'transactionNumber', 'transactionSeq_Total',
       'paymentType', 'paymentTransRef', 'gross_total'],
      dtype='object')

In [8]:
pivoted_sorted_df_paymentType = sorted_df_paymentType.pivot(index=['cinemaCode', 'transactionNumber',
                                                                   'transactionSeq_Total', 'paymentTransRef'], columns='', values='gross_total').reset_index()
# pivoted_sorted_df_paymentType.head()
filled_pivoted_sorted_df_paymentType = pivoted_sorted_df_paymentType.fillna(0)

**Merge between ....**

In [9]:
df_merge = sorted_df_boco_detail.merge(
    pivoted_sorted_df_paymentType, how='left', on=['cinemaCode', 'transactionNumber'])
df_merge.head()

,cinemaCode,transactionNumber,transactionSeq_Detail,transactionRefundSeq,closingDate,redeemDate,transactionDate,realTransactionDate,productHOCode,productName,...,PAYOO PAY,SHOPEE,UNKNOWN,VNPAY,VNPAY-QR,VNPAYGRS,Web Item Vouchers,ZALO,ZALOPAYGRS,ZaloPay-QR
0,0004,7708669,2,NaN,2025-05-02 00:58:26.137,2025-05-01 12:15:00,2025-04-23 16:46:55.617,2025-04-23 16:46:03,HO2DTHAMTK,THAM TU KIEN: KY AN KHONG DAU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0004,7708669,1,NaN,2025-05-02 00:58:26.137,2025-05-01 12:15:00,2025-04-23 16:46:55.617,2025-04-23 16:46:03,HO2DTHAMTK,THAM TU KIEN: KY AN KHONG DAU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0004,7708843,1,NaN,2025-05-02 00:58:26.137,2025-05-01 19:30:00,2025-04-23 19:04:02.923,2025-04-23 19:03:32,HO2DLAMAT8,LAT MAT 8: VONG TAY NANG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0004,7708843,4,NaN,2025-05-02 00:58:26.137,2025-05-01 19:30:00,2025-04-23 19:04:02.923,2025-04-23 19:03:32,HO2DLAMAT8,LAT MAT 8: VONG TAY NANG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0004,7708843,3,NaN,2025-05-02 00:58:26.137,2025-05-01 19:30:00,2025-04-23 19:04:02.923,2025-04-23 19:03:32,HO2DLAMAT8,LAT MAT 8: VONG TAY NANG,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
filled_df_merge = df_merge.fillna(0)
filled_df_merge.columns

Index(['cinemaCode', 'transactionNumber', 'transactionSeq_Detail',
       'transactionRefundSeq', 'closingDate', 'redeemDate', 'transactionDate',
       'realTransactionDate', 'productHOCode', 'productName', 'typeProduct',
       'itemHOCode', 'itemName', 'quantity', 'unitPrice', 'donViTinh',
       'gross_detail', 'tax', 'net', 'status', 'transactionSeq_Total',
       'paymentTransRef', 'APPMOMO', 'APPVNPAY', 'APPZALO', 'Cash', 'HSBC-ATM',
       'HSBC-VISA/MASTER', 'Loyalty Points', 'MOMO', 'MOMO PAY',
       'ONEPAY_OFF_QR', 'ONEPAY_ONL_ND', 'ONEPAY_ONL_QR', 'ONEPAY_ONL_QT',
       'PAYOO PAY', 'SHOPEE', 'UNKNOWN', 'VNPAY', 'VNPAY-QR', 'VNPAYGRS',
       'Web Item Vouchers', 'ZALO', 'ZALOPAYGRS', 'ZaloPay-QR'],
      dtype='object')

In [11]:
df = pd.DataFrame()

for trans_number in pivoted_sorted_df_paymentType['transactionNumber'].unique():
    df_test = sorted_df_boco_detail[sorted_df_boco_detail['transactionNumber'] == trans_number]

    payment_methods = [filled_pivoted_sorted_df_paymentType[filled_pivoted_sorted_df_paymentType['transactionNumber'] == trans_number][paymentType].sum()
                       for paymentType in df_receipt_cash_paymentType['paymentType'].unique()]


    values_transactions = df_test['gross_detail'].values


    allocation_result = allocate_payments(values_transactions, payment_methods)

    columns = ["transactionNumber"]+["Transaction"] + \
        [paymentType for paymentType in df_receipt_cash_paymentType['paymentType'].unique()]

    data = [[trans_number]+[values_transactions[i]] + allocation_result[i]
            for i in range(len(values_transactions))]
    # print(data)

    df = pd.concat([df, pd.DataFrame(data, columns=columns)],
                   ignore_index=True)

df

C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empt

C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empt

C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empt

C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empt

C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empt

C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empt

... (6 outputs left out)

C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empt

C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empt

C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(data, columns=columns)],
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\3828144047.py:22: FutureWarning: The behavior of DataFrame concatenation with empt

,transactionNumber,Transaction,APPMOMO,VNPAYGRS,APPVNPAY,SHOPEE,UNKNOWN,MOMO,HSBC-ATM,ONEPAY_ONL_QT,...,ZALO,ONEPAY_ONL_ND,Cash,Web Item Vouchers,ONEPAY_OFF_QR,VNPAY-QR,MOMO PAY,Loyalty Points,PAYOO PAY,ZaloPay-QR
0,7726359,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-35.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,7726359,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,7726359,65.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,7726359,-100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,7726359,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13472,7737277,89.0,89.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13473,7737281,109.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13474,7737282,89.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13475,7737286,109.0,109.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
df_drop = df.drop('Transaction', axis=1)

**Process PaymentType**

In [13]:
# Create a null row
sum_gross_detail = pd.DataFrame([df_drop.drop('transactionNumber', axis=1).sum()], columns=df_drop.columns)

# Concatenate the null row with the original DataFrame
df_Payment_final = pd.concat([sum_gross_detail, df_drop], ignore_index=True)
df_Payment_final['transactionNumber'] = df_Payment_final['transactionNumber'].fillna(0000)

# Create a null row to paymentType
null_row = pd.DataFrame([[np.nan] * len(sorted_df_boco_detail.columns)], columns=sorted_df_boco_detail.columns)

# Concatenate the null row with the original DataFrame
df_with_null_cash_paymentType = pd.concat([null_row, sorted_df_boco_detail], ignore_index=True)

df_with_null_cash_paymentType['transactionNumber'] = df_with_null_cash_paymentType['transactionNumber'].fillna(0000)


C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\1463244058.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_Payment_final['transactionNumber'] = df_Payment_final['transactionNumber'].fillna(0000)
C:\Users\liendtk.GALAXY\AppData\Local\Temp\ipykernel_8032\1463244058.py:12: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_with_null_cash_paymentType = pd.concat([null_row, sorted_df_boco_detail], ignore_index=True)


**concat between detail vs total**

In [14]:
# Filter rows with matching 'TransNumber'
common_numbers = set(df_with_null_cash_paymentType['transactionNumber']) & set(df_Payment_final['transactionNumber'])
df1_filtered = df_with_null_cash_paymentType[df_with_null_cash_paymentType['transactionNumber'].isin(common_numbers)]
df2_filtered = df_Payment_final[df_Payment_final['transactionNumber'].isin(common_numbers)]

# Concatenate filtered DataFrames horizontally
result = pd.concat([df1_filtered.reset_index(drop=True), 
                    df2_filtered.reset_index(drop=True)], axis=1)

In [ ]:
result

**save to file excel**

In [15]:
# result.to_excel('./datasets/Data_Invoice_Transaction_2804_0405_2025.xlsx',
#                 index=False, sheet_name='Thiso_SaLa')

In [16]:
# with pd.ExcelWriter('./datasets/Data_Invoice_Transaction_2804_0405_2025.xlsx', mode="a", engine="openpyxl") as writer:
#     # use to_excel function and specify the sheet_name and index to
#     # store the dataframe in specified sheet
#     result.to_excel(writer, sheet_name='KDV', index=False)